In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
def merge_to_silver(df, target_table, primary_key):
    if not spark.catalog.tableExists(target_table):

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

    else:

        delta_table = DeltaTable.forName(
            spark, target_table
        )

        (
            delta_table.alias("target")
            .merge(
                df.alias("source"),
                f"target.{primary_key} = source.{primary_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    print(f"Incremental load completed: {target_table}")

In [0]:
batch_id = "batch_002"

customers_batch = (
    spark.table("bank_catalog.bronze.customers")
    .filter(F.col("batch_id") == batch_id)
)

In [0]:
customers_clean = (
    customers_batch
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn(
        "date_of_birth",
        F.expr("try_cast(date_of_birth AS DATE)")
    )
    .withColumn(
        "onboarding_date",
        F.expr("try_cast(onboarding_date AS DATE)")
    )
    .withColumn(
        "customer_segment",
        F.upper(F.trim("customer_segment"))
    )
    .filter(
        F.col("customer_id").isNotNull() &
        F.col("date_of_birth").isNotNull()
    )
    .dropDuplicates(["customer_id"])
)

In [0]:
merge_to_silver(
    customers_clean,
    "bank_catalog.silver.customers",
    "customer_id"
)

In [0]:
def merge_to_gold(df, target_table, primary_key):

    if not spark.catalog.tableExists(target_table):

        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

    else:

        delta_table = DeltaTable.forName(
            spark, target_table
        )

        (
            delta_table.alias("target")
            .merge(
                df.alias("source"),
                f"target.{primary_key} = source.{primary_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

In [0]:
batch_id = "batch_002"

transactions_batch = (
    spark.table("bank_catalog.silver.transactions")
    .filter(F.col("batch_id") == batch_id)
)

accounts_all = spark.table("bank_catalog.silver.accounts")

fact_transaction_batch = (
    transactions_batch.alias("t")
    .join(
        accounts_all.alias("a"),
        F.col("t.account_id") == F.col("a.account_id"),
        "inner"
    )
    .select(
        F.col("t.transaction_id"),
        F.col("t.account_id"),
        F.col("a.customer_id"),
        F.col("a.branch_id"),
        F.date_format(
            F.col("t.transaction_timestamp"), "yyyyMMdd"
        ).cast("int").alias("date_key"),
        F.col("t.transaction_timestamp"),
        F.col("t.transaction_type"),
        F.col("t.amount"),
        F.col("t.currency"),
        F.col("t.channel"),
        F.col("t.transaction_status"),
        F.lit(1).alias("transaction_count")
    )
)

merge_to_gold(
    fact_transaction_batch,
    "bank_catalog.gold.fact_transaction",
    "transaction_id"
)

In [0]:
def log_batch(batch_id, table_name, status,
              records=0, error=None):

    audit_df = spark.createDataFrame(
        [(batch_id, table_name, status, records, error)],
        ["batch_id", "table_name", "status",
         "records_processed", "error_message"]
    )

    audit_df = (
        audit_df
        .withColumn("start_time", F.current_timestamp())
        .withColumn(
            "end_time",
            F.when(
                F.col("status") == "STARTED",
                F.lit(None).cast("timestamp")
            ).otherwise(F.current_timestamp())
        )
        .withColumn("event_timestamp", F.current_timestamp())
    )

    audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("bank_catalog.bronze.batch_audit")

In [0]:
%sql
SELECT *
FROM bank_catalog.bronze.batch_audit
ORDER BY event_timestamp DESC;

In [0]:
dbutils.widgets.text("batch_id", "batch_002")

batch_id = dbutils.widgets.get("batch_id")

print("Processing batch:", batch_id)

In [0]:
%run ./bank_ingestion

In [0]:
ingest_batch(batch_id)

In [0]:
ingest_batch("batch_002")